#### Imports

In [1]:
import  numpy as np

In [2]:
# Load both parts
dat0 = np.load('../data/synthetic/steinmetz_part0.npz', allow_pickle=True)['dat']
dat1 = np.load('../data/synthetic/steinmetz_part1.npz', allow_pickle=True)['dat']


In [3]:
# Combine all sessions
all_sessions = np.concatenate([dat0, dat1])
print(f"Total sessions: {len(all_sessions)}")

Total sessions: 26


session mean experiments ts What stimulus was shown to the mouse

In [4]:
s = all_sessions[0]
print(f"\nSession 0: {s['mouse_name']}, {s['date_exp']}")
print(f"Neurons: {s['spks'].shape[0]}")
print(f"Trials: {s['spks'].shape[1]}")
print(f"Time bins: {s['spks'].shape[2]}")
print(f"Bin size: {s['bin_size']}s")
print(f"Brain areas: {np.unique(s['brain_area'])}")
print(f"Response values: {np.unique(s['response'])}")


Session 0: Cori, 2016-12-14
Neurons: 734
Trials: 214
Time bins: 250
Bin size: 0.01s
Brain areas: ['ACA' 'CA3' 'DG' 'LS' 'MOs' 'SUB' 'VISp' 'root']
Response values: [-1.  0.  1.]


Session is just a number (0 to 25). Each session is one recording. Scientists stuck electrodes into a mouse's brain, showed it visual stimuli on screens, and recorded everything the neurons did. They did this 26 separate times.

Mouse is the name of the actual mouse. There are 7 mice: Cori, Forssmann, Hench, Lederberg, Moniz, Muller, Radnitz. Some mice were recorded multiple times on different days. Cori has 3 sessions, Lederberg has 7 sessions. Same mouse, different days, slightly different neural patterns each time.

Neurons is how many brain cells the electrode picked up in that session. Session 3 (Forssmann) captured 1769 neurons firing at once. Session 15 (Lederberg) only caught 474. The number varies because the electrode lands in slightly different spots each time.

Trials is how many times the mouse did the task in that session. One trial goes like this: two screens light up (left and right) with different brightness levels. The mouse looks at the screens, decides which side is brighter, and turns a wheel left or right to answer. One trial takes about 2.5 seconds. Session 9 (Hench) did 447 trials. Session 20 (Moniz) only did 124.

Regions is how many different brain areas the electrode passed through. A Neuropixels probe is a long thin needle that goes deep into the brain and records from multiple areas at once. Session 7 (Hench) hit 15 different brain regions in one recording. Some are visual areas (VISp), some are decision areas (MOs), some are memory areas (CA3, DG).

In [7]:
print(f"{'sessions': <8} {'mouse': <12} {'Neurons: >10'} {'Trials: >8}'} {'Area'}")
print("-" * 70)
for i, s in enumerate(all_sessions):
    n_neurons = s['spks'].shape[0]
    n_trials = s['spks'].shape[1]
    area = len(np.unique(s['brain_area']))
    print(f"{i:<8} {s['mouse_name']} {n_neurons:<10} {n_trials:<8} {area} regions")


sessions mouse        Neurons: >10 Trials: >8} Area
----------------------------------------------------------------------
0        Cori 734        214      8 regions
1        Cori 1070       251      5 regions
2        Cori 619        228      11 regions
3        Forssmann 1769       249      11 regions
4        Forssmann 1077       254      10 regions
5        Forssmann 1169       290      5 regions
6        Forssmann 584        252      8 regions
7        Hench 1156       250      15 regions
8        Hench 788        372      12 regions
9        Hench 1172       447      13 regions
10       Hench 857        342      6 regions
11       Lederberg 698        340      12 regions
12       Lederberg 983        300      15 regions
13       Lederberg 756        268      10 regions
14       Lederberg 743        404      8 regions
15       Lederberg 474        280      6 regions
16       Lederberg 565        224      6 regions
17       Lederberg 1089       316      10 regions
18       Moniz 6

In [10]:
# Pick a session to examine closely
s = all_sessions[0]

# What the mouse saw and did
contrast_left = s["contrast_left"]
contrast_right = s["contrast_right"]
response = s["response"]

print("contrast Level shows")
print(f"contrast left: {np.unique(contrast_left)}")
print(f"contrast right: {np.unique(contrast_right)}")

print("The response Distribution")

# -1 mean mouse chose left
# 0 mean he did non
# 1 mean he chose left
for r in [-1, 0, 1]:
    count = np.sum(response == r)
    pct = count / len(response) * 100
    print(f" {r:>2}: {count} trials ({pct:.1f}%)")

# The ML target: predict response from neural activity
# Remove no-response trials (mouse didn't engage

# Remove trials where mouse didn’t respond
active = response != 0
print(f"\nActive trials (mouse responded): {np.sum(active)} / {len(response)}")
print(f"Left vs Right split: {np.sum(response == -1)} left, {np.sum(response == 1)} right")



contrast Level shows
contrast left: [0.   0.25 0.5  1.  ]
contrast right: [0.   0.25 0.5  1.  ]
The response Distribution
 -1: 69 trials (32.2%)
  0: 74 trials (34.6%)
  1: 71 trials (33.2%)

Active trials (mouse responded): 140 / 214
Left vs Right split: 69 left, 71 right


Contrast values [0, 0.25, 0.5, 1.0] are brightness levels. 0 means screen is dark, 1 means full brightness. Each trial shows one brightness on the left screen and one on the right. Mouse has to pick the brighter side.

In [11]:
# Quick check  hoe meny usable trials we have across all the trials
total_active = 0
total_trials = 0

for s in all_sessions:
    total_active += len(s['response'])
    total_trials += len(s['response'])

print(f"Total active trials: {total_active}")
print(f"Total trials: {total_trials}")


Total active trials: 7108
Total trials: 7108


In [13]:
# lets see 0 responses
total_zero = 0
total_all = 0
for i, s in enumerate(all_sessions):
    resp = s['response']
    zeros = int(np.sum(resp == 0))
    total_zero += zeros
    total_all += len(resp)
    print(f"Session {i}: {len(resp)} trials, {zeros} no-response")

print(f"\nTotal trials: {total_all}")
print(f"No-response trials: {total_zero}")
print(f"Usable trials: {total_all - total_zero}")

Session 0: 214 trials, 74 no-response
Session 1: 251 trials, 94 no-response
Session 2: 228 trials, 77 no-response
Session 3: 249 trials, 126 no-response
Session 4: 254 trials, 104 no-response
Session 5: 290 trials, 92 no-response
Session 6: 252 trials, 62 no-response
Session 7: 250 trials, 93 no-response
Session 8: 372 trials, 134 no-response
Session 9: 447 trials, 155 no-response
Session 10: 342 trials, 97 no-response
Session 11: 340 trials, 64 no-response
Session 12: 300 trials, 56 no-response
Session 13: 268 trials, 61 no-response
Session 14: 404 trials, 97 no-response
Session 15: 280 trials, 74 no-response
Session 16: 224 trials, 52 no-response
Session 17: 316 trials, 76 no-response
Session 18: 247 trials, 112 no-response
Session 19: 235 trials, 94 no-response
Session 20: 124 trials, 56 no-response
Session 21: 444 trials, 115 no-response
Session 22: 151 trials, 65 no-response
Session 23: 187 trials, 83 no-response
Session 24: 261 trials, 76 no-response
Session 25: 178 trials, 50 no

4869 usable trials where the mouse actually made a decision

In [14]:
# Feature extraction: mean firing rate per neuron in 4 time windows
# Each trial is 250 bins at 10ms = 2.5 seconds
# Stimulus appears around bin 50 (0.5s)

# Time windows (in bins):
# Pre-stimulus:  0-49    (before anything happens)
# Stimulus:      50-99   (visual processing)
# Decision:      100-149 (mouse deciding)
# Post-decision: 150-249 (after choice)


windows = {
    'pre_stim': (0,50),
    'stimulus': (50,100),
    'decisions': (100,150),
    'post_dec': (150,200)
}

'''
| Window      |  Bin range |   Time range | Meaning                 |
| ----------- | ---------: | -----------: | ----------------------- |
| `pre_stim`  |    0 to 49 | 0.0s to 0.5s | before stimulus         |
| `stimulus`  |   50 to 99 | 0.5s to 1.0s | visual input processing |
| `decisions` | 100 to 149 | 1.0s to 1.5s | mouse deciding          |
| `post_dec`  | 150 to 199 | 1.5s to 2.0s | after choice            |

'''

# Extract from one session first to test
s = all_sessions[0]
spks = s['spks']           # shape: (734, 214, 250)
resp = s['response']       # shape: (214,)

# Keep only active trials
active_mask = resp != 0
spks_active = spks[:, active_mask, :]   # (734, 140, 250)
labels = resp[active_mask]              # (140,)

# Convert labels: -1 -> 0, 1 -> 1
labels = (labels + 1) // 2             # now 0 = left, 1 = right
labels = labels.astype(int)

# Mean firing rate per neuron per window per trial
n_neurons = spks_active.shape[0]
n_trials = spks_active.shape[1]
n_features = n_neurons * len(windows)   # 734 * 4 = 2936 features

X = np.zeros((n_trials, n_features))
for i, (name, (start, end)) in enumerate(windows.items()):
    # Mean firing rate in this window for all neurons, all trials
    rates = spks_active[:, :, start:end].mean(axis=2)  # (734, 140)
    X[:, i*n_neurons:(i+1)*n_neurons] = rates.T         # (140, 734)

print(f"Feature matrix: {X.shape}")
print(f"Labels: {labels.shape}, {np.sum(labels==0)} left, {np.sum(labels==1)} right")
print(f"Features per trial: {n_features} (734 neurons x 4 windows)")

Feature matrix: (140, 2936)
Labels: (140,), 69 left, 71 right
Features per trial: 2936 (734 neurons x 4 windows)


Working perfectly. 140 trials, 2936 features each, nearly balanced classes.

In [18]:
# Build features from ALL sessions
X_all = []
y_all = []
session_ids = []

for i, s in enumerate(all_sessions):
    spks = s['spks']        # (neurons, trials, time_bins)
    resp = s['response']

    # Active trials only
    active_mask = resp != 0
    if np.sum(active_mask) == 0:
        continue

    spks_active = spks[:, active_mask, :]
    labels = ((resp[active_mask] + 1) // 2).astype(int)

    n_neurons = spks_active.shape[0]
    n_trials = spks_active.shape[1]
    n_bins = spks_active.shape[2]

    # Adjust windows to this session's time bins
    quarter = n_bins // 4
    local_windows = [
        (0, quarter),
        (quarter, 2*quarter),
        (2*quarter, 3*quarter),
        (3*quarter, n_bins)
    ]

    # Mean firing rate per window
    X = np.zeros((n_trials, n_neurons * 4))
    for j, (start, end) in enumerate(local_windows):
        rates = spks_active[:, :, start:end].mean(axis=2)  # (n_neurons, n_trials)
        X[:, j*n_neurons:(j+1)*n_neurons] = rates.T         # (n_trials, n_neurons)

    X_all.append(X)
    y_all.append(labels)
    session_ids.extend([i] * n_trials)

session_ids = np.array(session_ids)
y_all = np.concatenate(y_all)

print(f"Total usable trials: {len(y_all)}")
print(f"Labels: {np.sum(y_all==0)} left, {np.sum(y_all==1)} right")
print(f"Sessions represented: {len(np.unique(session_ids))}")
print(f"\nEach session has different neuron count:")
for i, x in enumerate(X_all):
    print(f"  Session {i}: {x.shape[0]} trials, {x.shape[1]} features ({x.shape[1]//4} neurons)")

Total usable trials: 4869
Labels: 2419 left, 2450 right
Sessions represented: 26

Each session has different neuron count:
  Session 0: 140 trials, 2936 features (734 neurons)
  Session 1: 157 trials, 4280 features (1070 neurons)
  Session 2: 151 trials, 2476 features (619 neurons)
  Session 3: 123 trials, 7076 features (1769 neurons)
  Session 4: 150 trials, 4308 features (1077 neurons)
  Session 5: 198 trials, 4676 features (1169 neurons)
  Session 6: 190 trials, 2336 features (584 neurons)
  Session 7: 157 trials, 4624 features (1156 neurons)
  Session 8: 238 trials, 3152 features (788 neurons)
  Session 9: 292 trials, 4688 features (1172 neurons)
  Session 10: 245 trials, 3428 features (857 neurons)
  Session 11: 276 trials, 2792 features (698 neurons)
  Session 12: 244 trials, 3932 features (983 neurons)
  Session 13: 207 trials, 3024 features (756 neurons)
  Session 14: 307 trials, 2972 features (743 neurons)
  Session 15: 206 trials, 1896 features (474 neurons)
  Session 16: 172

In [19]:
# SOLUTION: Train per-session, then aggregate
# This is actually how real BCI labs do it.
# Each session gets its own decoder.
# For the MLOps pipeline, this is BETTER because:
#   - Each session = a "deployment" that can drift independently
#   - Model registry stores one model per session
#   - Drift detection runs per session
#   - Retraining triggers per session
# This is more realistic than one giant model.

# But first, let's prove the concept works on one session.
# Train/test split on Session 0:

from numpy.random import default_rng

rng = default_rng(42)
X = X_all[0]
y = y_all[:140]

# Shuffle and split 80/20
idx = rng.permutation(len(y))
split = int(0.8 * len(y))
X_train, X_test = X[idx[:split]], X[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]

# Normalize features (zero mean, unit variance per feature)
mean = X_train.mean(axis=0)
std = X_train.std(axis=0) + 1e-8   # avoid divide by zero
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train labels: {np.sum(y_train==0)} left, {np.sum(y_train==1)} right")
print(f"Test labels: {np.sum(y_test==0)} left, {np.sum(y_test==1)} right")

# Simple baseline: logistic regression in pure NumPy
# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# Train with gradient descent
n_features = X_train.shape[1]
w = rng.normal(0, 0.01, n_features)
b = 0.0
lr = 0.01

'''
This baseline feeds into the self-improving framework. Your framework adds self-observation, self-diagnosis, and self-correction on top of a basic training loop. It needs access to the raw weights, gradients, and per-neuron activation stats. Sklearn doesn't expose those internals cleanly. Pure NumPy gives you full control, which is exactly what the self-improving layer needs to detect and fix dead neurons.
'''
for epoch in range(200):
    # Forward
    z = X_train @ w + b
    pred = sigmoid(z)

    # Loss
    loss = -np.mean(y_train * np.log(pred + 1e-8) + (1 - y_train) * np.log(1 - pred + 1e-8))

    # Gradients
    error = pred - y_train
    dw = X_train.T @ error / len(y_train)
    db = np.mean(error)

    # Update
    w -= lr * dw
    b -= lr * db

# Evaluate
test_pred = (sigmoid(X_test @ w + b) > 0.5).astype(int)
accuracy = np.mean(test_pred == y_test)
print(f"\nBaseline accuracy: {accuracy:.3f}")
print(f"Chance level: {max(np.mean(y_test), 1-np.mean(y_test)):.3f}")

Train: (112, 2936), Test: (28, 2936)
Train labels: 57 left, 55 right
Test labels: 12 left, 16 right

Baseline accuracy: 0.643
Chance level: 0.571


In [20]:
# Baseline accuracy across ALL sessions
print(f"{'Session':<8} {'Trials':<8} {'Neurons':<8} {'Accuracy':<10} {'Chance':<8}")
print("-" * 50)

accuracies = []
for i, X in enumerate(X_all):
    # Get labels for this session
    mask = session_ids == i
    y = y_all[mask]

    # Shuffle and split
    idx = rng.permutation(len(y))
    split = int(0.8 * len(y))
    X_tr, X_te = X[idx[:split]], X[idx[split:]]
    y_tr, y_te = y[idx[:split]], y[idx[split:]]

    # Normalize
    mu = X_tr.mean(axis=0)
    sd = X_tr.std(axis=0) + 1e-8
    X_tr = (X_tr - mu) / sd
    X_te = (X_te - mu) / sd

    # Train logistic regression
    nf = X_tr.shape[1]
    w = rng.normal(0, 0.01, nf)
    b = 0.0

    for epoch in range(300):
        z = X_tr @ w + b
        pred = sigmoid(z)
        error = pred - y_tr
        w -= 0.01 * (X_tr.T @ error / len(y_tr))
        b -= 0.01 * np.mean(error)

    # Evaluate
    te_pred = (sigmoid(X_te @ w + b) > 0.5).astype(int)
    acc = np.mean(te_pred == y_te)
    chance = max(np.mean(y_te), 1 - np.mean(y_te))
    accuracies.append(acc)

    print(f"{i:<8} {len(y):<8} {nf//4:<8} {acc:<10.3f} {chance:<8.3f}")

print(f"\nMean accuracy: {np.mean(accuracies):.3f}")
print(f"Sessions above chance: {sum(1 for a in accuracies if a > 0.55)}/26")

Session  Trials   Neurons  Accuracy   Chance  
--------------------------------------------------
0        140      734      0.750      0.571   
1        157      1070     0.688      0.625   
2        151      619      0.806      0.613   
3        123      1769     0.440      0.520   
4        150      1077     0.633      0.567   
5        198      1169     0.675      0.550   
6        190      584      0.842      0.605   
7        157      1156     0.719      0.562   
8        238      788      0.792      0.604   
9        292      1172     0.814      0.644   
10       245      857      0.837      0.612   
11       276      698      0.875      0.518   
12       244      983      0.857      0.592   
13       207      756      0.833      0.595   
14       307      743      0.871      0.516   
15       206      474      0.905      0.571   
16       172      565      0.829      0.514   
17       240      1089     0.604      0.521   
18       135      606      0.926      0.519   
19       

In [22]:
# Save the extraction config and results for reference
import json

results = {
    'total_sessions': 26,
    'total_usable_trials': int(len(y_all)),
    'label_balance': {'left': int(np.sum(y_all==0)), 'right': int(np.sum(y_all==1))},
    'baseline_mean_accuracy': round(float(np.mean(accuracies)), 3),
    'sessions_above_chance': int(sum(1 for a in accuracies if a > 0.55)),
    'per_session': [
        {
            'session': i,
            'trials': int(X_all[i].shape[0]),
            'neurons': int(X_all[i].shape[1] // 4),
            'baseline_accuracy': round(float(accuracies[i]), 3)
        }
        for i in range(26)
    ]
}

with open('../data/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved baseline_results.json")

Saved baseline_results.json
